# 1117 53일차

## 1. 조건 인덱싱

비교식의 결과로 행을 고르는 것 (`pd06_loc_iloc3_condition`)

값을 보고 고를 때는 행 이름이나 번호를 미리 알 수 없어서, 조건을 적어 맞는 행만 남김

1. `df['시가'] >= 1100` : 행마다 True/False가 든 Series가 나옴
2. `df[cond1]` : True인 행만 남김
3. `df[df['시가'] >= 1100]['종가']` : 거른 뒤 열을 하나 꺼냄

- **대괄호에 True/False Series를 넣으면 행을 고르는 것으로 해석됨**
- 조건식을 변수에 담아 두면 조건과 거르는 자리를 나눠 볼 수 있음
- 크기를 비교하려면 그 열이 숫자여야 함. 따옴표를 씌워 두면 글자 순서로 비교됨
- 거른 결과도 DataFrame이라 뒤에 `['종가']`를 붙여 열을 꺼냄
- `df[조건].loc['종가']`는 `loc`이 행을 보는 자리라 `KeyError`

## 2. 이상치

다른 값들과 크게 떨어져 있는 값 (`pd07_outlier1`, `pd07_outlier2`)

평균이나 분산을 쓰는 계산은 값 하나에 통째로 끌려가므로, 학습 전에 찾아 둬야 함

1. `np.percentile(data, [25, 50, 75])` : 1사분위·중위·3사분위를 한 번에 받음
2. `iqr = quartile_3 - quartile_1` : 가운데 50%가 퍼져 있는 폭
3. `lower_bound = quartile_1 - (iqr * 1.5)`, `upper_bound = quartile_3 + (iqr * 1.5)` : 이상치로 보는 경계
4. `np.where((data > upper_bound) | (data < lower_bound))` : 경계 밖 값의 위치
5. `plt.boxplot(data)` : 사분위와 경계를 그림으로 보여 주고, 경계 밖 값은 점으로 찍음

- **사분위수는 순위로 뽑는 값이라 극단값이 끼어도 자리만 밀릴 뿐 흔들리지 않음**
- 1.5는 관습으로 쓰는 배수. 3으로 키우면 덜 걸림
- 조건마다 괄호를 쳐야 함. `|`가 비교보다 먼저 묶여서 에러가 남
- `np.where`에 조건만 넣으면 인덱스가 든 튜플이 나와서 `[0]`으로 꺼냄
- 값이 넓게 퍼진 열은 경계도 넓게 잡혀, 동떨어져 보여도 안 걸릴 수 있음
- 2차원은 행마다 기준이 달라서 따로 구함

#### cf) 박스 플롯 그리기

- `plt.boxplot`에 2차원을 넣으면 열마다 박스를 그림. 행마다 그리려면 전치해서 넣음
- 범위가 다른 데이터를 한 축에 올리면 작은 쪽이 눌려 보이지 않아 `plt.subplot`으로 나눔
- `plt.axhline`은 축 전체를 가로질러서, 서브플롯으로 나눠야 경계선이 자기 박스에만 그려짐
- `label`만 주면 범례가 나오지 않고 `plt.legend()`를 불러야 함

## 3. 이상치 처리

찾은 이상치를 결측치로 바꿔 채우는 것 (`pd07_outlier2`)

이상치는 값이 들어 있는 상태라 `dropna`로는 지워지지 않음. NaN으로 바꾸면 결측치 처리 방법을 그대로 씀

1. `pd.DataFrame(aaa.T, columns=['x1', 'x2'])` : 행이 데이터, 열이 특성이 되게 전치해서 담음
2. `df.loc[outlier_loc1[0], 'x1'] = np.nan` : 찾아 둔 위치의 값을 NaN으로 바꿈
3. `df.interpolate()` 다음 `df.bfill()` : 가운데를 보간으로 채우고 남은 양 끝을 채움

- **이상치를 NaN으로 바꾸면 그다음은 결측치와 같은 문제가 됨**
- `bfill`을 먼저 부르면 양 끝이 채워져서 보간이 들어갈 자리가 없어짐
- 행을 통째로 버리면 그 행의 다른 열 값까지 같이 버려짐
- IQR 경계 안이어도 그 열에 나올 수 없는 값이면 위치를 직접 적어 바꿈
- for문으로 하나씩 지우면 지울 때마다 뒤 인덱스가 당겨져 엉뚱한 자리를 지움